In [1]:
# @title 0. PURPOSE:
# This script unzips the AffectNet dataset into the appropriate directory for further processing.
print(
    "**************************************************************************************\n"
    "The scripts import AffectNet dataset and unzip it into the 'data/AffectNet' directory.\n"
    "**************************************************************************************"
)

**************************************************************************************
The scripts import AffectNet dataset and unzip it into the 'data/AffectNet' directory.
**************************************************************************************


## 0. Setup configuration

In [2]:
# 0. Install packages and Check

# a. Packages Installation

# # Kaggle Accelarator
# Selectionner 'ACCELERATOR': GPU T4 x2

# # Package à inistaller utiliser le GPU de Kaggle et Ultralytics
# pip install numpy -U --force-reinstall
# pip install matplotlib -U --force-reinstall
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# pip install --upgrade ultralytics

In [3]:
# 1. Load requested libraires

# Pour pouvoir utiliser le TPU de Kaggle et vérifier la version
import torch
import ultralytics

import os
import kagglehub
import shutil
import yaml

from ultralytics import YOLO
from IPython.display import display

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
# Quick checking: Vérifier que PyTorch et Ultralytics sont bien installés
print(torch.__version__)
print(ultralytics.__version__)

2.6.0+cu124
8.3.233


In [5]:
!nvidia-smi

Sun Nov 30 23:45:37 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# 2. Afficher les  GPU disponible, nombre et nom du premier
print("CUDA disponible :", torch.cuda.is_available())
print("Nombre de GPUs :", torch.cuda.device_count())
print("Nom du GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")

CUDA disponible : True
Nombre de GPUs : 2
Nom du GPU : Tesla T4


## I. Download AffectNet dataset

In [7]:
# 1. Download dataset 'AffectNet'
path = kagglehub.dataset_download("fatihkgg/affectnet-yolo-format")
print(f"Path to dataset files: {path}")

Path to dataset files: /kaggle/input/affectnet-yolo-format


In [8]:
# 3. Define parameters
# Current Directory
HOME = os.getcwd()
print(f"Message: Curent working directory is set to {HOME}")

# Parameters
DATASET_NAME = "AffectNet"

# The source path for the YOLO_format data
source_data_path = os.path.join(path, "YOLO_format")

# The user-specified destination path on Google Drive
destination_path = os.path.join(HOME, "data", "YOLO_format")

file_path = "/kaggle/working/data/YOLO_format/data_AffectNet.yaml"

Message: Curent working directory is set to /kaggle/working


In [9]:
# # 2. Copy data to a specific destination

# # The source path for the YOLO_format data
# source_data_path = os.path.join(path, "YOLO_format")

# # The user-specified destination path on Google Drive
# destination_path = os.path.join(HOME, "data", "YOLO_format")

# # Create destination directory if it doesn't exist
# os.makedirs(destination_path, exist_ok=True)

# # If a previous copy exists, remove it
# if os.path.exists(destination_path):
#     shutil.rmtree(destination_path)

# # Copy the folder recursively
# shutil.copytree(source_data_path, destination_path)

# print(f"Copie terminée ! Données copiées dans : {destination_path}")


In [10]:
# 3. Quick check: Cardinality of extract files
print("Cadinality of extracted files:")

def count_files_in_directory(destination_path):
  """Count files in a given directory."""
  # Ensure the directory exists before trying to list its contents
  if not os.path.isdir(destination_path):
      print(f"Warning: Directory not found: {destination_path}")
      return 0
  return len([f for f in os.listdir(destination_path) if os.path.isfile(os.path.join(destination_path, f))])

# List of directories to check (assuming YOLO_format is a subfolder)
directories = [
    os.path.join(destination_path, "test", "images"),
    os.path.join(destination_path, "train", "images"),
    os.path.join(destination_path, "valid", "images")
]

# Display results
sets = ["Train", "Test", "Valid"]

for dir_path, set_name in zip(directories, sets):
  file_count = count_files_in_directory(dir_path)
  print(f"Message: {set_name} --> {file_count} files")

Cadinality of extracted files:
Message: Train --> 2755 files
Message: Test --> 17101 files
Message: Valid --> 5406 files


## II. Create YAML - AffectNet

In [11]:
# Définir le contenu YAML
data_AffectNet_yaml = {
    "path": "/kaggle/working/data/YOLO_format",
    "train": "/kaggle/working/data/YOLO_format/train/images",
    "val": "/kaggle/working/data/YOLO_format/valid/images",
    "test": "/kaggle/working/data/YOLO_format/test/images",
    
    "nc": 8,
    "names": [
        "Anger",
        "Contempt",
        "Disgust",
        "Fear",
        "Happy",
        "Neutral",
        "Sad",
        "Surprise"
    ],
    
    # Network cofiguration (choose other variations if desired)
    "network": "yolov8m"
}


In [12]:
# Sauvegarder dans un fichier
file_path = "/kaggle/working/data/YOLO_format/data_AffectNet.yaml"

with open(file_path, "w") as f:
    yaml.dump(data_AffectNet_yaml, f)
    
print(f"Fichier YAML sauvegardé à : {file_path}")

Fichier YAML sauvegardé à : /kaggle/working/data/YOLO_format/data_AffectNet.yaml


In [13]:
# Vérification rapide de l'existence des path
print(os.path.exists("/kaggle/working/data/YOLO_format/valid/images"))
print(os.listdir("/kaggle/working/data/YOLO_format"))


True
['data.yaml', 'train', 'test', 'data_AffectNet.yaml', 'valid']


## III. Train Emotion model with YOLO 

In [14]:
# 1. Charger un modèle YOLOv8 pré-entraîné
model = YOLO("yolov8m.pt")

In [15]:
# 2. Entraînement
results = model.train(
    data=file_path,
    epochs=50,          # Utiliser 100 epochs après
    imgsz=640,
    batch=16,
    device=0,           # Remplace "gpu" par "cpu" ou "0" si GPU dispo
    lr0=0.001,              # learning rate
    optimizer="Adam",       # optimiser
    augment=True,           # activer les augmentations par défaut
    project="affectnet_training",
    name="yolov8m_emotions_v1"
)

Ultralytics 8.3.233 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data/YOLO_format/data_AffectNet.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8m_emotions_v118, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patience=100

In [20]:
# 3. Évaluation automatique sur le set de validation
metrics = model.val()

Ultralytics 8.3.233 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 448.9±287.4 MB/s, size: 14.2 KB)
val: Scanning /kaggle/working/data/YOLO_format/valid/labels.cache... 5406 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5406/5406 7.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 338/338 2.3it/s 2:25<0.4s
                   all       5406       5406      0.697      0.757        0.8        0.8
                 Anger        712        712      0.668      0.803        0.8        0.8
              Contempt        618        618      0.705      0.782      0.831      0.831
               Disgust        672        672      0.667      0.701      0.758      0.758
                  Fear        622        622      0.692      0.746      0.824      0.824
                 Happy        791        791      0.869      0.869      0.951      0.951
               N

In [21]:
# 4. Test sur le set test
model.val(split="test")

Ultralytics 8.3.233 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 319.0±197.8 MB/s, size: 8.6 KB)
val: Scanning /kaggle/working/data/YOLO_format/test/labels.cache... 2755 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2755/2755 5.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 173/173 2.3it/s 1:150.4sss
                   all       2755       2755      0.706      0.757      0.804      0.804
                 Anger        383        383      0.691      0.794      0.807      0.807
              Contempt        332        332       0.71      0.773      0.797      0.797
               Disgust        327        327      0.679      0.674      0.758      0.758
                  Fear        318        318      0.746      0.768      0.845      0.845
                 Happy        399        399      0.857      0.884      0.945      0.945
               Ne

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b9807e56910>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

In [22]:
# 5. Export du modèle (ONNX, other like TorchScript, etc., is desired)
model.export(format="onnx")

Ultralytics 8.3.233 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon CPU @ 2.00GHz)

PyTorch: starting from '/kaggle/working/affectnet_training/yolov8m_emotions_v118/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 12, 8400) (49.6 MB)

ONNX: starting export with onnx 1.18.0 opset 19...
ONNX: slimming with onnxslim 0.1.77...
ONNX: export success ✅ 2.8s, saved as '/kaggle/working/affectnet_training/yolov8m_emotions_v118/weights/best.onnx' (98.8 MB)

Export complete (4.2s)
Results saved to /kaggle/working/affectnet_training/yolov8m_emotions_v118/weights
Predict:         yolo predict task=detect model=/kaggle/working/affectnet_training/yolov8m_emotions_v118/weights/best.onnx imgsz=640  
Validate:        yolo val task=detect model=/kaggle/working/affectnet_training/yolov8m_emotions_v118/weights/best.onnx imgsz=640 data=/kaggle/working/data/YOLO_format/data_AffectNet.yaml  
Visualize:       https://netron.app


'/kaggle/working/affectnet_training/yolov8m_emotions_v118/weights/best.onnx'

In [24]:
# 6. Sauvegarde du modèle entraîné
model_dir = os.path.join(HOME, "models")
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(HOME, "models", "yolov8m_emotions_affectnet.pt")
model.save(model_path)
print(f"Modèle sauvegardé sous: {model_path}")

Modèle sauvegardé sous: /kaggle/working/models/yolov8m_emotions_affectnet.pt


In [ ]:
# 7. Quick check: Prédiction de mon modèle sur une image quelconque

from IPython.display import display

img_path = ''
image_files = [os.path.join(img_path, f"img{i}.png") for i in range(1, 6)]

# Prédiction
check_results = [model.predict(source=image, save=True, conf=0.5) for image in image_files]

# Affichage des résultats
for i, result in enumerate(check_results):
    print(f"Résultat pour {image_files[i]} :")
    display(result[0].plot())